In [1]:
import pandas as pd
import numpy as np

In [2]:
xls = pd.ExcelFile("Annotation_SynPCC7942_RND.xlsx") #seperates xlsx file into seperate dataframes for each sheet
ncbi   = pd.read_excel(xls, 'NCBI')

ncbi=ncbi.rename(columns=ncbi.iloc[0]).loc[1:] #gets rid of first row of dataframe to make second row the header
ncbi.set_index("locus_tag",inplace=True) #resets index to be locus tags
ncbi.sort_index(key=lambda x: x.str.lower(), inplace = True) #organizes locus tags by numerical orders
ncbi.columns = ncbi.columns.fillna('to_drop') #renames all Nan columsn to be "to_drop"
ncbi_no_nan=ncbi.drop('to_drop', axis = 1) #drops all columns named "to_drop", they didn't want to drop  with nan label

cyto=pd.read_csv("grn_1100_min9_03_25_24.csv") #importing data table got from cytoscape
cyto.rename(columns={"name" : "locus_tag"}, inplace=True) #renames name column to locus tags to merge on later
cyto.set_index("locus_tag", inplace = True) #sets index as locus tags
cyto.sort_index(key=lambda x: x.str.lower(), inplace = True) #again reorganizes them in numerical order


In [3]:
ncbi_merge=pd.merge(ncbi_no_nan, cyto, how="left", on=["locus_tag"])
tf_merge=ncbi_merge.query("TF == 1")

In [7]:
#creates excel document with the two seperate sheets
#need to add top most row back to these sheets from the original file
with pd.ExcelWriter("grn_1100_min9_annotation_03_25_24.xlsx") as writer:
    ncbi_merge.to_excel(writer, sheet_name="NCBI Annotation ", index=True)
    tf_merge.to_excel(writer, sheet_name = "Transcription regulators")
